<a href="https://colab.research.google.com/github/mr-zero-000/Statistical-Learning-e23034/blob/main/Assignment%209/Question_04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gaussian Mixture Clustering as Conditional Updating

---

## Task 1: Deriving the Marginal Density

### 1. Proof of the Marginal Density Formula

We want to derive the marginal density $p(x_i)$ of an observation $x_i$ using the law of total probability.

**Proof:**
By the law of total probability for continuous random variables conditioned on a discrete latent space $C_i \in \{1, \dots, K\}$, the marginal density is the sum over all possible mutual states:

$$p(x_i) = \sum_{k=1}^K p(x_i, C_i = k)$$

Using the definition of joint probability via conditional probability ($p(A, B) = p(A \mid B)P(B)$):

$$p(x_i) = \sum_{k=1}^K p(x_i \mid C_i = k) P(C_i = k)$$

We are given the following explicit components:

1. The prior probability of cluster membership: $P(C_i = k) = \phi_k$
2. The conditional distribution: $X_i \mid C_i = k \sim \mathscr{N}(\mu_k, \Sigma_k)$, which yields the density function $p(x_i \mid C_i = k) = \mathscr{N}(x_i \mid \mu_k, \Sigma_k)$

Substituting these two terms directly into the summation expression yields:

$$p(x_i) = \sum_{k=1}^K \phi_k \mathscr{N}(x_i \mid \mu_k, \Sigma_k) \quad \re$$

### 2. Definition of a Gaussian Mixture Density

This expression is called a **Gaussian mixture density** because it forms a convex combination (a linear blend) of $K$ distinct multivariate normal distributions. Each base Gaussian component density $\mathscr{N}(x_i \mid \mu_k, \Sigma_k)$ is weighted by its respective prior probability $\phi_k$. Because $\phi_k \ge 0$ and $\sum_{k=1}^K \phi_k = 1$, the overall integration over the entire space equals exactly $1.0$, rendering it a mathematically sound, unified probability distribution.

---

## Task 2: Deriving the Posterior Cluster Probability

### 1. Proof of the Posterior Relation (Bayes' Rule)

We want to derive the conditional probability that an individual point $x_i$ belongs to cluster $k$.

**Proof:**
Applying Bayes' Theorem directly to the discrete random variable $C_i$ conditioned on the observed continuous variable $X_i = x_i$:

$$P(C_i = k \mid X_i = x_i) = \frac{p(X_i = x_i, C_i = k)}{p(x_i)}$$

Expanding both the numerator (via the product rule) and the denominator (via the law of total probability derived in Task 1):

$$P(C_i = k \mid X_i = x_i) = \frac{p(X_i = x_i \mid C_i = k)P(C_i = k)}{\sum_{j=1}^K p(X_i = x_i \mid C_i = j)P(C_i = j)} \quad \b$$

Substituting the explicit distribution parameters $P(C_i = k) = \phi_k$ and $p(x_i \mid C_i = k) = \mathscr{N}(x_i \mid \mu_k, \Sigma_k)$ directly into the equation yields the standard definition of **responsibility** ($\gamma_{ik}$):

$$\gamma_{ik} = P(C_i = k \mid X_i = x_i) = \frac{\phi_k \mathscr{N}(x_i \mid \mu_k, \Sigma_k)}{\sum_{j=1}^K \phi_j \mathscr{N}(x_i \mid \mu_j, \Sigma_j)} \quad \b$$

### 2. Interpretation as a Posterior Probability

The value $\gamma_{ik}$ represents a true posterior probability because it updates our prior belief ($\phi_k$) after incorporating the physical evidence provided by the location of $x_i$. The term $\mathscr{N}(x_i \mid \mu_k, \Sigma_k)$ acts as a spatial compatibility metric. If a data point lies close to the center $\mu_k$, this component likelihood becomes large, dynamically increasing the updated probability $\gamma_{ik}$ relative to the alternative clusters.

---

## Task 3: One-Hot Encoding of the Latent Cluster Variable

### 1. Proof of Identity for the Single Components

We want to show that the conditional expectation of the binary indicator $Z_{ik}$ is identical to the posterior probability of cluster assignment.

**Proof:**
By the mathematical definition of conditional expectation for a discrete random variable:

$$\mathbb{E}[Z_{ik} \mid X_i = x_i] = \sum_{z \in \{0, 1\}} z \cdot P(Z_{ik} = z \mid X_i = x_i)$$

Expanding the summation across its two possible discrete states ($z=0$ and $z=1$):

$$\mathbb{E}[Z_{ik} \mid X_i = x_i] = \left( 0 \cdot P(Z_{ik} = 0 \mid X_i = x_i) \right) + \left( 1 \cdot P(Z_{ik} = 1 \mid X_i = x_i) \right)$$

$$\mathbb{E}[Z_{ik} \mid X_i = x_i] = P(Z_{ik} = 1 \mid X_i = x_i)$$

By the definition of our one-hot encoding configuration, $Z_{ik} = 1$ if and only if $C_i = k$. Therefore, the probability states are identical:

$$\mathbb{E}[Z_{ik} \mid X_i = x_i] = P(C_i = k \mid X_i = x_i) \quad \e$$

### 2. Vector Proof and Expectation Conclusion

Using the linearity property of expectations, applying the conditional operator component-wise across the full vector $\mathbf{Z}_i$ yields:

$$\mathbb{E}[\mathbf{Z}_i \mid X_i = x_i] = \begin{bmatrix}  \mathbb{E}[Z_{i1} \mid X_i = x_i] \\  \mathbb{E}[Z_{i2} \mid X_i = x_i] \\  \vdots \\  \mathbb{E}[Z_{iK} \mid X_i = x_i]  \end{bmatrix} = \begin{bmatrix}  P(C_i = 1 \mid X_i = x_i) \\  P(C_i = 2 \mid X_i = x_i) \\  \vdots \\  P(C_i = K \mid X_i = x_i)  \end{bmatrix} = \begin{bmatrix}  \gamma_{i1} \\  \gamma_{i2} \\  \vdots \\  \gamma_{iK}  \end{bmatrix} \quad \be$$

This proves that the soft cluster assignment vector in a Gaussian mixture model is precisely the **conditional expectation of the one-hot encoded latent variable** given the observed data point.

---

## Task 4: From Soft Assignment to Hard Clustering

| Clustering Mode | Assignment Property | Mathematical Definition | Characteristic Behavior |
| --- | --- | --- | --- |
| **Soft Clustering** | Probabilistic fractional membership | $\mathbb{E}[\mathbf{Z}_i \mid X_i = x_i] = [\gamma_{i1}, \dots, \gamma_{iK}]^T$ | Expresses ambiguity by distributing weights across multiple overlapping groups. |
| **Hard Clustering** | Deterministic binary assignment | $\widehat{C}_i = \operatorname{arg\,max}_{1 \le k \le K} \gamma_{ik}$ | Eliminates uncertainty by forcing a point into a single cluster based on the Maximum A Posteriori (MAP) decision rule. |

---

## Task 5: Conditional Expectation of the Observation Given the Cluster

### 1. Proof of the Spatial Mean Identity

We want to prove that the conditional expectation of $X_i$ given the cluster assignment $C_i = k$ is exactly $\mu_k$.

**Proof:**
Given that $X_i \mid C_i = k \sim \mathscr{N}(\mu_k, \Sigma_k)$, the conditional density function is a standard multivariate Gaussian:

$$\mathbb{E}[X_i \mid C_i = k] = \int_{\mathbb{R}^d} x_i \cdot \mathscr{N}(x_i \mid \mu_k, \Sigma_k) \, dx_i$$

By applying a linear shift substitution $u = x_i - \mu_k$ (implying $dx_i = du$):

$$\mathbb{E}[X_i \mid C_i = k] = \int_{\mathbb{R}^d} (u + \mu_k) \cdot \mathscr{N}(u \mid \mathbf{0}, \Sigma_k) \, du$$

$$\mathbb{E}[X_i \mid C_i = k] = \left( \int_{\mathbb{R}^d} u \cdot \mathscr{N}(u \mid \mathbf{0}, \Sigma_k) \, du \right) + \left( \mu_k \int_{\mathbb{R}^d} \mathscr{N}(u \mid \mathbf{0}, \Sigma_k) \, du \right)$$

Because a zero-mean Gaussian density is a symmetric function, the first integral evaluates to exactly $\mathbf{0}$. Since a normalized probability density integrates to $1$, the second integral simplifies to $\mu_k \cdot 1$. Thus:

$$\mathbb{E}[X_i \mid C_i = k] = \mathbf{0} + \mu_k = \mu_k \quad \ble$$

Consequently, $\mu_k$ represents the central coordinate space (centroid) of the $k$-th cluster.

### 2. Comparison of the Two Conditional Expectations

* **$\mathbb{E}[\mathbf{Z}_i \mid X_i = x_i]$ (Soft Membership):** Maps from the continuous observation space $\mathbb{R}^d$ to the probability simplex $\Delta^{K-1}$. It identifies **where a data point belongs** across the cluster options.
* **$\mathbb{E}[X_i \mid C_i = k]$ (Cluster Center):** Maps from the discrete latent label space to the continuous coordinate space $\mathbb{R}^d$. It identifies **where the cluster itself is located** in the physical feature space.

---

## Task 6: The Complete-Data Likelihood

### 1. Proof of the Complete-Data Log-Likelihood Form

We want to show that taking the natural logarithm of the complete-data likelihood function results in the standard additive formulation $\ell_c$.

**Proof:**
We start with the product formulation of the complete-data likelihood:

$$p(x_1, \dots, x_n, z_1, \dots, z_n) = \prod_{i=1}^n \prod_{k=1}^K \left[ \phi_k \mathscr{N}(x_i \mid \mu_k, \Sigma_k) \right]^{z_{ik}}$$

Applying the natural logarithm to both sides:

$$\ell_c = \log \left( \prod_{i=1}^n \prod_{k=1}^K \left[ \phi_k \mathscr{N}(x_i \mid \mu_k, \Sigma_k) \right]^{z_{ik}} \right)$$

Using the logarithmic identity $\log(\prod A_m) = \sum \log(A_m)$, we convert the products into summations:

$$\ell_c = \sum_{i=1}^n \sum_{k=1}^K \log \left( \left[ \phi_k \mathscr{N}(x_i \mid \mu_k, \Sigma_k) \right]^{z_{ik}} \right)$$

Using the logarithmic power identity $\log(A^B) = B\log(A)$, we bring the indicators down:

$$\ell_c = \sum_{i=1}^n \sum_{k=1}^K z_{ik} \log \left[ \phi_k \mathscr{N}(x_i \mid \mu_k, \Sigma_k) \right]$$

Finally, expanding the inner product term into an additive formulation ($\log(AB) = \log A + \log B$):

$$\ell_c = \sum_{i=1}^n \sum_{k=1}^K z_{ik} \left[ \log \phi_k + \log \mathscr{N}(x_i \mid \mu_k, \Sigma_k) \right] \quad \e$$

### 2. Optimization Advantage of the Complete-Data Likelihood

If the values of $z_{ik}$ were known, maximizing this objective function would be straightforward. Because the indicator variable $z_{ik}$ acts as a binary selector ($0$ or $1$), the log-likelihood decouples completely across the components. Instead of dealing with sums inside logarithms—as seen in the marginal likelihood—we could optimize each cluster's parameters $(\mu_k, \Sigma_k)$ independently using only the data points assigned to that specific cluster.

---

## Task 7: The EM Interpretation

In practice, the binary values $z_{ik}$ are hidden. The **Expectation Maximization (EM) algorithm** bypasses this by calculating the conditional expectation of the complete log-likelihood given the observed data and current parameter estimates $\boldsymbol{\theta}^{(t)}$. This yields the $Q$-function:

$$Q(\boldsymbol{\theta} \mid \boldsymbol{\theta}^{(t)}) = \mathbb{E}_{Z \mid X, \boldsymbol{\theta}^{(t)}}[\ell_c] = \sum_{i=1}^n \sum_{k=1}^K \mathbb{E}[Z_{ik} \mid X_i = x_i, \boldsymbol{\theta}^{(t)}] \left[ \log \phi_k + \log \mathscr{N}(x_i \mid \mu_k, \Sigma_k) \right]$$

Substituting the conditional expectation identity proved in Task 3 ($\mathbb{E}[Z_{ik} \mid X_i = x_i] = \gamma_{ik}$):

$$Q = \sum_{i=1}^n \sum_{k=1}^K \gamma_{ik} \left[ \log \phi_k + \log \mathscr{N}(x_i \mid \mu_k, \Sigma_k) \right] \quad \e$$

The E-step can be viewed as a **conditional update of cluster membership probabilities**. The algorithm evaluates the model's current parameters, looks at where the data points fall, and uses Bayes' rule to update the membership probabilities (responsibilities) for every point across all clusters.

---

## Task 8: Parameter Updates

We can derive the M-step updates by maximizing the $Q$-function with respect to each parameter.

### 1. Verification of the Mixture Weights Update ($\phi_k$)

We want to maximize $Q$ with respect to $\phi_k$, subject to the constraint that $\sum_{k=1}^K \phi_k = 1$. We construct the Lagrangian function with multiplier $\lambda$:

$$\mathcal{L}(\phi, \lambda) = \left( \sum_{i=1}^n \sum_{k=1}^K \gamma_{ik} \log \phi_k \right) - \lambda \left( \sum_{k=1}^K \phi_k - 1 \right)$$

Taking the partial derivative with respect to a specific cluster weight $\phi_k$ and setting it to zero:

$$\frac{\partial \mathcal{L}}{\partial \phi_k} = \sum_{i=1}^n \frac{\gamma_{ik}}{\phi_k} - \lambda = 0 \implies \sum_{i=1}^n \gamma_{ik} = \lambda \phi_k$$

Let $N_k = \sum_{i=1}^n \gamma_{ik}$ represent the effective number of points assigned to cluster $k$. Then $N_k = \lambda \phi_k$. Summing both sides over all $K$ clusters to isolate $\lambda$:

$$\sum_{k=1}^K N_k = \lambda \sum_{k=1}^K \phi_k \implies \sum_{k=1}^K \sum_{i=1}^n \gamma_{ik} = \lambda (1)$$

Since the responsibilities for any point sum to 1 ($\sum_{k=1}^K \gamma_{ik} = 1$), the double sum simplifies to $n$:

$$\sum_{i=1}^n 1 = n \implies \lambda = n$$

Substituting $\lambda = n$ back into the derivative equation yields the optimal update:

$$\phi_k^{\text{new}} = \frac{N_k}{n} \quad \e$$

### 2. Verification of the Cluster Mean Update ($\mu_k$)

To maximize $Q$ with respect to the center vector $\mu_k$, we expand the Gaussian term and focus on the components that depend on $\mu_k$:

$$Q_{\mu_k} = \sum_{i=1}^n \gamma_{ik} \left( -\frac{1}{2}(x_i - \mu_k)^T \Sigma_k^{-1}(x_i - \mu_k) \right)$$

Taking the vector derivative with respect to $\mu_k$ using the chain rule:

$$\frac{\partial Q}{\partial \mu_k} = \sum_{i=1}^n \gamma_{ik} \Sigma_k^{-1}(x_i - \mu_k) = 0$$

Since the covariance matrix $\Sigma_k^{-1}$ is non-singular, we can multiply through by $\Sigma_k$ to remove it:

$$\sum_{i=1}^n \gamma_{ik}(x_i - \mu_k) = \mathbf{0} \implies \sum_{i=1}^n \gamma_{ik}x_i = \left(\sum_{i=1}^n \gamma_{ik}\right)\mu_k$$

Substituting our definition for the effective number of points ($N_k = \sum_{i=1}^n \gamma_{ik}$) and isolating $\mu_k$ yields the update formula:

$$\mu_k^{\text{new}} = \frac{1}{N_k} \sum_{i=1}^n \gamma_{ik}x_i \quad \e$$

### 3. Verification of the Covariance Update ($\Sigma_k$)

Similarly, maximizing the $Q$-function with respect to the covariance matrix $\Sigma_k$ yields the weighted sample covariance:

$$\Sigma_k^{\text{new}} = \frac{1}{N_k} \sum_{i=1}^n \gamma_{ik}(x_i - \mu_k^{\text{new}})(x_i - \mu_k^{\text{new}})^T \quad \e$$

### 4. Interpretation of Fractional Weights

The responsibility $\gamma_{ik}$ acts as a **fractional membership weight**. Instead of a data point contributing fully to only one cluster, it contributes a fraction of its presence to each cluster based on $\gamma_{ik}$. If $\gamma_{ik} = 0.85$, then $85\%$ of the point's value is used to compute the updated mean $\mu_k$ and covariance $\Sigma_k$, allowing the model to smoothly handle overlapping data regions.

---

## Task 9: Interpretation

Gaussian Mixture Model (GMM) clustering can be understood as an iterative process of conditional updating. The model parameters $\phi_k$, $\mu_k$, and $\Sigma_k$ represent our current assumptions about the clusters, where the mixture weight $\phi_k$ defines the prior probability of a point belonging to cluster $k$. When an observation $x_i$ is introduced, the multivariate Gaussian density $\mathscr{N}(x_i \mid \mu_k, \Sigma_k)$ calculates its spatial compatibility with each group.

By combining these prior probabilities and component likelihoods via Bayes' rule, the E-step computes the responsibilities $\gamma_{ik}$. As established in Task 3, these responsibilities form the soft assignment vector $\mathbb{E}[\mathbf{Z}_i \mid X_i = x_i]$, which represents the conditional expectation of the latent cluster membership variable.

During the M-step, the algorithm updates the cluster parameters using these posterior probabilities as fractional weights. This two-step cycle forms a feedback loop: updated cluster positions alter the conditional expectations of cluster membership, which in turn adjust the cluster locations. Ultimately, Gaussian mixture clustering is a fully probabilistic approach grounded in the conditional expectations of latent variables.

---



In [ ]:
## Interactive Implementation in Python

'''Run this complete script in your notebook to see the GMM conditional updates in action on a synthetic dataset.

python'''
import numpy as np
import plotly.graph_objects as go
from scipy.stats import multivariate_normal

# Set seed for reproducible simulation results
np.random.seed(42)

# --- 1. Generate Synthetic Mixture Data ---
num_samples = 150
true_mu1, true_sigma1 = np.array([1.0, 1.0]), np.array([[0.3, 0.1], [0.1, 0.3]])
true_mu2, true_sigma2 = np.array([3.5, 3.5]), np.array([[0.5, -0.1], [-0.1, 0.4]])

data1 = np.random.multivariate_normal(true_mu1, true_sigma1, num_samples)
data2 = np.random.multivariate_normal(true_mu2, true_sigma2, num_samples)
X = np.vstack((data1, data2))

# --- 2. Initialize GMM Parameters ---
K = 2
n, d = X.shape
phi = np.array([0.5, 0.5])
mu = np.array([[0.0, 2.0], [4.0, 2.0]])  # Intentional initial offset
sigma = np.array([np.eye(d), np.eye(d)])

# --- 3. Execute a Single E-Step (Conditional Update) ---
responsibilities = np.zeros((n, K))

for k in range(K):
    # Compute the likelihood component pointwise
    likelihood = multivariate_normal.pdf(X, mean=mu[k], cov=sigma[k])
    responsibilities[:, k] = phi[k] * likelihood

# Normalize across rows to get valid posterior distributions (γ_ik)
row_sums = responsibilities.sum(axis=1, keepdims=True)
responsibilities /= row_sums

# --- 4. Visualize the Probabilistic Soft Assignments ---
fig = go.Figure()

# Color points based on their posterior responsibility for Cluster 1
fig.add_trace(go.Scatter(
    x=X[:, 0], y=X[:, 1],
    mode='markers',
    marker=dict(
        size=8,
        color=responsibilities[:, 0],
        colorscale='Viridis',
        showscale=True,
        colorbar=dict(title="Responsibility (γ_i1)")
    ),
    name='Data Points'
))

# Plot the current cluster centers (μ_k)
fig.add_trace(go.Scatter(
    x=mu[:, 0], y=mu[:, 1],
    mode='markers+text',
    marker=dict(size=14, color='red', symbol='x'),
    text=[f"Center μ1", f"Center μ2"],
    textposition="top center",
    name='Cluster Centroids'
))

fig.update_layout(
    title="GMM Soft Assignments: Conditional Expectation E[Z_i | X_i]",
    xaxis_title="Feature Dimension 1",
    yaxis_title="Feature Dimension 2",
    template="plotly_white"
)
fig.show()

```